# EfficientNet-B0 
- Epoch 1 완료 | Train Acc: 97.00%
- Epoch 2 완료 | Train Acc: 99.30%
- Epoch 3 완료 | Train Acc: 99.75%

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# [수정 1] 과적합 방지를 위한 강력한 데이터 증강
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),       # 이미지를 랜덤하게 자름
    transforms.RandomHorizontalFlip(),       # 좌우 반전
    transforms.RandomRotation(15),           # 15도 내외 회전
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# [수정 2] 모델 구조 변경 (Dropout 추가)
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=4)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, 512), # [추가] 512개의 노드를 가진 은닉층
    nn.ReLU(),                                   # [추가] 비선형성 부여
    nn.Dropout(0.2),                              # [추가] 추가적인 과적합 방지
    nn.Linear(512, 4)                             # 최종 출력층
)

# [수정 3] Weight Decay 강화 (AdamW 사용)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-1) 
criterion = nn.CrossEntropyLoss()

# 이후 학습 루프는 이전과 동일하게 사용하시면 됩니다.

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# [수정 1] 과적합 방지를 위한 강력한 데이터 증강
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),       # 이미지를 랜덤하게 자름
    transforms.RandomHorizontalFlip(),       # 좌우 반전
    transforms.RandomRotation(15),           # 15도 내외 회전
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# [수정 2] 모델 구조 변경 (Dropout 추가)
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=4)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, 1024), # 노드 증가
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(1024, 512), # 은닉층 추가 (2단 구조)
    nn.ReLU(),
    nn.Linear(512, 4)
)

# [수정 3] Weight Decay 강화 (AdamW 사용)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-1) 
criterion = nn.CrossEntropyLoss()

# 이후 학습 루프는 이전과 동일하게 사용하시면 됩니다.

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# [수정 1] 과적합 방지를 위한 강력한 데이터 증강
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),       # 이미지를 랜덤하게 자름
    transforms.RandomHorizontalFlip(),       # 좌우 반전
    transforms.RandomRotation(15),           # 15도 내외 회전
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# [수정 2] 모델 구조 변경 (Dropout 추가)
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=4)
# [실험 코드] 난이도가 높은 강력한 데이터 증강 적용
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)), # 더 공격적으로 자름
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8), # 색상 변형 추가
    transforms.RandomGrayscale(p=0.2), # 랜덤 흑백화 추가
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), # 회전각 증가
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# [수정 3] Weight Decay 강화 (AdamW 사용)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-1) 
criterion = nn.CrossEntropyLoss()

# 이후 학습 루프는 이전과 동일하게 사용하시면 됩니다.

In [19]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 3
MAX_TRAIN_SAMPLES = 500        # ⚡ 알아서 클래스당 딱 500장만 채우고 정지!
MAX_VAL_SAMPLES = 50           # ⚡ 알아서 클래스당 딱 50장만 채우고 정지!

print(f"🚀 [초고속 폴더 기반 스캔 시스템] 가동 (장치: {DEVICE})")

DATA_ROOT = Path(r"C:\Project\PyCharmMiscProject\PythonProject02\organized_bee_dataset")
TRAIN_IMAGE_DIR = DATA_ROOT / "train"
VAL_IMAGE_DIR = DATA_ROOT / "val_train"

# ==========================================
# ⚡ JSON을 무겁게 열지 않고, 폴더 구조에서 선착순으로 가져오는 정석 데이터셋
# ==========================================
class FastBeeDataset(Dataset):
    def __init__(self, image_dir, transform=None, max_samples=500, mode="train"):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        
        # 0:알, 1:백묵병, 2:유충_응애, 3:유충_정상
        self.class_to_idx = {"알": 0, "백묵병": 1, "유충_응애": 2, "유충_정상": 3}
        
        print(f"🔍 [{mode.upper()}] 폴더에서 선착순 {max_samples}장씩 자동 수집 중...")
        
        for class_name, idx in self.class_to_idx.items():
            class_folder = image_dir / class_name
            if class_folder.exists():
                # 폴더 내 모든 이미지를 빠르게 리스트업
                all_images = [p for p in class_folder.glob("*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
                
                # ⚡ [핵심] 컴퓨터가 알아서 딱 500장(혹은 50장)만 슬라이싱해서 가져옵니다! 뒤에는 쳐다보지도 않음.
                limited_images = all_images[:max_samples]
                
                for img_path in limited_images:
                    self.image_paths.append(img_path)
                    self.labels.append(idx)
                
                print(f"   - {class_name}: {len(limited_images)}장 확보완료")
        print(f"📦 [{mode.upper()}] 총 {len(self.image_paths)}장 세팅 완료! (대기 시간 없음)\n")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
            label = self.labels[idx]
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception:
            return self.__getitem__((idx + 1) % len(self.image_paths))

# ==========================================
# 3. 모델 및 로더 연결 (동일)
# ==========================================
model = timm.create_model('hf_hub:timm/efficientnet_b0.ra_in1k', pretrained=True, num_classes=4)
model = model.to(DEVICE)
transform = timm.data.create_transform(**timm.data.resolve_model_data_config(model))



# ⚡ 무거운 연산 없이 즉시 실행됩니다.
train_dataset = FastBeeDataset(TRAIN_IMAGE_DIR, transform=transform, max_samples=MAX_TRAIN_SAMPLES, mode="train")
val_dataset = FastBeeDataset(VAL_IMAGE_DIR, transform=transform, max_samples=MAX_VAL_SAMPLES, mode="val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 4. 고속 학습 루프 (검증 과정 제거)
# ==========================================
print(f"🚀 [검증 없이] 학습을 시작합니다.")

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    # ⚡ train_loader만 사용합니다.
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        train_pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(train_correct/train_total)*100:.2f}%")
        
    # Validation 루프(if문 포함)를 모두 삭제했습니다.
    print(f"📊 Epoch {epoch+1} 완료 | Train Acc: {(train_correct/train_total)*100:.2f}%\n")

# 최종 모델 저장
torch.save(model.state_dict(), 'optimized_bee_model_no_val.pth')
print(f"💾 학습 완료! 'optimized_bee_model_no_val.pth' 저장 성공!")

🚀 [초고속 폴더 기반 스캔 시스템] 가동 (장치: cuda)
🔍 [TRAIN] 폴더에서 선착순 500장씩 자동 수집 중...
   - 알: 500장 확보완료
   - 백묵병: 500장 확보완료
   - 유충_응애: 500장 확보완료
   - 유충_정상: 500장 확보완료
📦 [TRAIN] 총 2000장 세팅 완료! (대기 시간 없음)

🔍 [VAL] 폴더에서 선착순 50장씩 자동 수집 중...
   - 알: 50장 확보완료
   - 백묵병: 50장 확보완료
   - 유충_응애: 50장 확보완료
   - 유충_정상: 50장 확보완료
📦 [VAL] 총 200장 세팅 완료! (대기 시간 없음)

🚀 [검증 없이] 학습을 시작합니다.


Epoch 1/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:43<00:00,  3.55s/it, acc=96.20%, loss=0.0467]


📊 Epoch 1 완료 | Train Acc: 96.20%



Epoch 2/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:42<00:00,  3.52s/it, acc=99.75%, loss=0.0000]


📊 Epoch 2 완료 | Train Acc: 99.75%



Epoch 3/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [02:49<00:00,  2.70s/it, acc=99.95%, loss=0.0006]

📊 Epoch 3 완료 | Train Acc: 99.95%

💾 학습 완료! 'optimized_bee_model_no_val.pth' 저장 성공!


In [22]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 3
MAX_TRAIN_SAMPLES = 500        # ⚡ 알아서 클래스당 딱 500장만 채우고 정지!
MAX_VAL_SAMPLES = 50           # ⚡ 알아서 클래스당 딱 50장만 채우고 정지!

print(f"🚀 [초고속 폴더 기반 스캔 시스템] 가동 (장치: {DEVICE})")

DATA_ROOT = Path(r"C:\Project\PyCharmMiscProject\PythonProject02\organized_bee_dataset")
TRAIN_IMAGE_DIR = DATA_ROOT / "train"
VAL_IMAGE_DIR = DATA_ROOT / "val_train"

# ==========================================
# ⚡ JSON을 무겁게 열지 않고, 폴더 구조에서 선착순으로 가져오는 정석 데이터셋
# ==========================================
class FastBeeDataset(Dataset):
    def __init__(self, image_dir, transform=None, max_samples=500, mode="train"):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        
        # 0:알, 1:백묵병, 2:유충_응애, 3:유충_정상
        self.class_to_idx = {"알": 0, "백묵병": 1, "유충_응애": 2, "유충_정상": 3}
        
        print(f"🔍 [{mode.upper()}] 폴더에서 선착순 {max_samples}장씩 자동 수집 중...")
        
        for class_name, idx in self.class_to_idx.items():
            class_folder = image_dir / class_name
            if class_folder.exists():
                # 폴더 내 모든 이미지를 빠르게 리스트업
                all_images = [p for p in class_folder.glob("*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
                
                # ⚡ [핵심] 컴퓨터가 알아서 딱 500장(혹은 50장)만 슬라이싱해서 가져옵니다! 뒤에는 쳐다보지도 않음.
                limited_images = all_images[:max_samples]
                
                for img_path in limited_images:
                    self.image_paths.append(img_path)
                    self.labels.append(idx)
                
                print(f"   - {class_name}: {len(limited_images)}장 확보완료")
        print(f"📦 [{mode.upper()}] 총 {len(self.image_paths)}장 세팅 완료! (대기 시간 없음)\n")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
            label = self.labels[idx]
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception:
            return self.__getitem__((idx + 1) % len(self.image_paths))

# ==========================================
# 3. 모델 및 로더 연결 (동일)
# ==========================================
model = timm.create_model('hf_hub:timm/efficientnet_b0.ra_in1k', pretrained=True, num_classes=4)
model = model.to(DEVICE)
transform = timm.data.create_transform(**timm.data.resolve_model_data_config(model))



# ⚡ 무거운 연산 없이 즉시 실행됩니다.
train_dataset = FastBeeDataset(TRAIN_IMAGE_DIR, transform=transform, max_samples=MAX_TRAIN_SAMPLES, mode="train")
val_dataset = FastBeeDataset(VAL_IMAGE_DIR, transform=transform, max_samples=MAX_VAL_SAMPLES, mode="val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=5e-1)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 4. 고속 학습 루프 (검증 과정 제거)
# ==========================================
print(f"🚀 [검증 없이] 학습을 시작합니다.")

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    # ⚡ train_loader만 사용합니다.
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        train_pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(train_correct/train_total)*100:.2f}%")
        
    # Validation 루프(if문 포함)를 모두 삭제했습니다.
    print(f"📊 Epoch {epoch+1} 완료 | Train Acc: {(train_correct/train_total)*100:.2f}%\n")

# 최종 모델 저장
torch.save(model.state_dict(), 'optimized_bee_model_no_val.pth')
print(f"💾 학습 완료! 'optimized_bee_model_no_val.pth' 저장 성공!")

🚀 [초고속 폴더 기반 스캔 시스템] 가동 (장치: cuda)
🔍 [TRAIN] 폴더에서 선착순 500장씩 자동 수집 중...
   - 알: 500장 확보완료
   - 백묵병: 500장 확보완료
   - 유충_응애: 500장 확보완료
   - 유충_정상: 500장 확보완료
📦 [TRAIN] 총 2000장 세팅 완료! (대기 시간 없음)

🔍 [VAL] 폴더에서 선착순 50장씩 자동 수집 중...
   - 알: 50장 확보완료
   - 백묵병: 50장 확보완료
   - 유충_응애: 50장 확보완료
   - 유충_정상: 50장 확보완료
📦 [VAL] 총 200장 세팅 완료! (대기 시간 없음)

🚀 [검증 없이] 학습을 시작합니다.


Epoch 1/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:02<00:00,  2.89s/it, acc=80.10%, loss=0.2004]


📊 Epoch 1 완료 | Train Acc: 80.10%



Epoch 2/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:44<00:00,  3.56s/it, acc=95.65%, loss=0.4717]


📊 Epoch 2 완료 | Train Acc: 95.65%



Epoch 3/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:36<00:00,  3.44s/it, acc=97.75%, loss=0.0092]

📊 Epoch 3 완료 | Train Acc: 97.75%

💾 학습 완료! 'optimized_bee_model_no_val.pth' 저장 성공!


In [2]:
import torch

# 파이썬 pickle 대신 PyTorch의 torch.load를 사용합니다.
# 파일명이 'optimized_bee_model_backup.pth'이므로 경로를 올바르게 맞춰줍니다.
model_data = torch.load('optimized_bee_model_no_val.pth', map_location='cpu')

print(model_data.keys())  # 내부 레이어 이름 확인

odict_keys(['conv_stem.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'blocks.0.0.conv_dw.weight', 'blocks.0.0.bn1.weight', 'blocks.0.0.bn1.bias', 'blocks.0.0.bn1.running_mean', 'blocks.0.0.bn1.running_var', 'blocks.0.0.bn1.num_batches_tracked', 'blocks.0.0.se.conv_reduce.weight', 'blocks.0.0.se.conv_reduce.bias', 'blocks.0.0.se.conv_expand.weight', 'blocks.0.0.se.conv_expand.bias', 'blocks.0.0.conv_pw.weight', 'blocks.0.0.bn2.weight', 'blocks.0.0.bn2.bias', 'blocks.0.0.bn2.running_mean', 'blocks.0.0.bn2.running_var', 'blocks.0.0.bn2.num_batches_tracked', 'blocks.1.0.conv_pw.weight', 'blocks.1.0.bn1.weight', 'blocks.1.0.bn1.bias', 'blocks.1.0.bn1.running_mean', 'blocks.1.0.bn1.running_var', 'blocks.1.0.bn1.num_batches_tracked', 'blocks.1.0.conv_dw.weight', 'blocks.1.0.bn2.weight', 'blocks.1.0.bn2.bias', 'blocks.1.0.bn2.running_mean', 'blocks.1.0.bn2.running_var', 'blocks.1.0.bn2.num_batches_tracked', 'blocks.1.0.se.conv_reduce.weigh

In [3]:
import torch
import timm

# 1. 모델 아키텍처 정의
# 저장된 파일의 구조와 정확히 일치하도록 num_classes를 설정해야 합니다.
# (파일이 4개 클래스용이므로 4로 지정)
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=4)

# 2. 가중치 파일 경로 설정
# 사용하고 싶은 파일명을 입력하세요 ('optimized_bee_model.pth' 또는 '..._backup.pth')
model_path = 'optimized_bee_model_no_val.pth' 

# 3. 가중치 로드
try:
    # map_location='cpu'를 사용하여 CPU 환경에서도 안전하게 불러옵니다.
    model_data = torch.load(model_path, map_location='cpu')
    
    # 모델에 가중치 입히기
    model.load_state_dict(model_data)
    print(f"성공: '{model_path}'의 가중치가 모델에 정상적으로 로드되었습니다.")
    
except Exception as e:
    print(f"오류 발생: 가중치 로드 실패 - {e}")

# 4. 모델 모드 설정
# 추론(테스트)을 위해 모델을 eval 모드로 전환합니다.
model.eval()

# 5. 확인 출력 (선택 사항)
# 모델이 정상적으로 로드되었는지 최종 확인
print("모델이 추론 준비 완료 상태입니다.")

# 이후 이 model 객체를 사용하여 실제 이미지를 예측(predict)할 수 있습니다.

성공: 'optimized_bee_model_no_val.pth'의 가중치가 모델에 정상적으로 로드되었습니다.
모델이 추론 준비 완료 상태입니다.


In [4]:
from PIL import Image
from timm.data import create_transform
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD

# 전처리 함수 정의 (이미지 크기 조절 및 정규화)
transform = create_transform(
    input_size=224, # efficientnet_b0의 기본 입력 사이즈
    mean=IMAGENET_DEFAULT_MEAN,
    std=IMAGENET_DEFAULT_STD
)
def predict_image(image_path, model):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0) # 배치 차원 추가 (1, 3, 224, 224)
    
    with torch.no_grad(): # 역전파 비활성화 (메모리 절약)
        output = model(img_tensor)
        prediction = torch.argmax(output, dim=1)
        
    return prediction.item()

# 사용 예시
image_path = 'test.jpg' # 실제 이미지 파일 경로
result = predict_image(image_path, model)
print(f"예측된 클래스 인덱스: {result}")

예측된 클래스 인덱스: 2


In [5]:
# 테스트할 이미지 파일 경로들을 리스트로 만듭니다.
# 예: 폴더별로 한 장씩 골라보세요.
test_images = ['test.jpg', 'test1.jpg', 'test2.jpg','test3.jpg']

for img_path in test_images:
    idx = predict_image(img_path, model)
    print(f"이미지 {img_path} -> 예측된 인덱스: {idx}")

이미지 test.jpg -> 예측된 인덱스: 2
이미지 test1.jpg -> 예측된 인덱스: 3
이미지 test2.jpg -> 예측된 인덱스: 1
이미지 test3.jpg -> 예측된 인덱스: 0


In [6]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 응애
확신도(Confidence): 100.00%
------------------------------


In [8]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test2.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 백묵병
확신도(Confidence): 100.00%
------------------------------


In [9]:
import cv2
import torch
import torch.nn.functional as F
import numpy as np
from PIL import ImageFont, ImageDraw, Image
from collections import deque

# 1. 환경 설정
cap = cv2.VideoCapture('백묵병.mp4')
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=30)
labels = ["알", "백묵병", "응애", "정상"]
fontpath = "malgun.ttf"
font = ImageFont.truetype(fontpath, 30)

# 경고 및 카운팅 변수
alert_threshold = 90.0
consecutive_frames = 10
warning_counter = 0
bee_count = 0
prev_centers = []

model.eval()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # --- [기능 2] 모델 분류 및 경고 알림 ---
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    img_tensor = transform(img_pil).unsqueeze(0)
    
    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output[0], dim=0).cpu().numpy()
        pred_idx = np.argmax(probs)
        confidence = probs[pred_idx] * 100

    # 경고 로직 (응애/백묵병 90% 이상 연속 10프레임)
    if (labels[pred_idx] in ["응애", "백묵병"]) and (confidence >= alert_threshold):
        warning_counter += 1
    else:
        warning_counter = 0

    # --- [기능 3] 화면 시각화 ---
    draw = ImageDraw.Draw(img_pil)
    # 상태 텍스트
    draw.text((50, 50), f"상태: {labels[pred_idx]} ({confidence:.1f}%)", font=font, fill=(0, 255, 0))

    # 경고 테두리
    if warning_counter >= consecutive_frames:
        draw.rectangle([0, 0, frame.shape[1]-1, frame.shape[0]-1], outline="red", width=10)
        draw.text((50, 150), "!!! 위험 감지 !!!", font=font, fill="red")

    # 출력
    cv2.imshow('Integrated System', cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR))
    if cv2.waitKey(30) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [10]:
import cv2
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image
import numpy as np
from collections import deque

# 1. 설정 및 초기화
# 웹캠이면 0, 영상 파일이면 '파일경로.mp4' 입력
video_source = 0  # <--- 여기에 파일명을 넣으세요. 예: 'bee_video.mp4'
cap = cv2.VideoCapture('백묵병.mp4') 

# 한글 폰트 설정 (폰트 파일 경로 확인!)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 30)
labels = ["알", "백묵병", "응애", "정상"]

# 평균을 내기 위한 버퍼 (최근 10개 프레임 저장)
prob_buffer = deque(maxlen=10)

model.eval()

print("분석을 시작합니다. 'q'를 누르면 종료됩니다.")

# 1. 이전 결과들을 저장할 바구니 (최근 15프레임)
history = deque(maxlen=15) 

while True:
    ret, frame = cap.read()
    if not ret: break

    # 2. 전처리
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    img_tensor = transform(img_pil).unsqueeze(0)

    # 3. 모델 추론
    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output[0], dim=0).cpu().numpy()
        history.append(probs) # 바구니에 저장

        # 2. 핵심: 최근 15개 프레임의 평균 확률 계산
        avg_probs = np.mean(history, axis=0)
        prediction = np.argmax(avg_probs)
        confidence = avg_probs[prediction] * 100
    
        # 3. 화면에 출력 (안정된 라벨)
        img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        draw = ImageDraw.Draw(img_pil)
        
        # 이제 라벨이 튀지 않고 평균적으로 확신하는 것만 나옵니다.
        text = f"{labels[prediction]} ({confidence:.1f}%)"
        draw.text((50, 50), text, font=font, fill=(0, 255, 0))
    
        cv2.imshow('Stable Analysis', cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR))
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
print("분석이 종료되었습니다.")

분석을 시작합니다. 'q'를 누르면 종료됩니다.
분석이 종료되었습니다.
